# 실습: Gemini API 기본 호출


## 수업 진행 방식

- 위에서 아래로 한 셀씩 실행합니다.
- API 요청이 발생하는 셀은 `[API 요청 n]` 형태로 요청 번호를 출력합니다.
- 429 `RESOURCE_EXHAUSTED`가 나오면 할당량 또는 짧은 시간 안의 요청량 문제일 수 있습니다.
- 수업 중에는 1~4번 요청까지만 실행해도 기본 개념을 충분히 확인할 수 있습니다.

## 0. 준비 확인

먼저 `.env` 파일에 아래 값이 들어 있어야 합니다.

```text
GEMINI_API_KEY=발급받은_API_KEY
```

`GEMINI_MODEL`을 지정하지 않으면 기본값으로 `gemini-3.5-flash`를 사용합니다.

In [9]:
import os
import time

from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError(".env 파일에 GEMINI_API_KEY=... 값을 넣어주세요.")

MODEL = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite")
REQUEST_DELAY_SECONDS = float(os.getenv("GEMINI_REQUEST_DELAY_SECONDS", "1.0"))

client = genai.Client(api_key=api_key)
api_request_count = 0

print(f"MODEL = {MODEL}")
print(f"REQUEST_DELAY_SECONDS = {REQUEST_DELAY_SECONDS}")

MODEL = gemini-3.1-flash-lite
REQUEST_DELAY_SECONDS = 1.0


## 1. 공통 도우미 함수

여러 셀에서 같은 방식으로 API를 호출하기 위해 작은 함수를 먼저 만들어 둡니다.

- 요청 번호를 출력합니다.
- 요청 사이에 짧은 간격을 둡니다.
- Gemini API 오류가 나면 `code`, `status`, `message`를 보기 쉽게 출력합니다.

In [10]:
def call_gemini(label, request_func):
    """Gemini API 요청을 보낼 때 요청 번호와 에러를 함께 관리한다."""
    global api_request_count
    api_request_count += 1

    print(f"\n[API 요청 {api_request_count}] {label}")
    print("-" * 50)

    if api_request_count > 1:
        time.sleep(REQUEST_DELAY_SECONDS)

    try:
        return request_func()
    except errors.APIError as exc:
        message = exc.message or str(exc)
        print(f"APIError [{exc.code} {exc.status}]: {message[:300]}")
        return None
    except Exception as exc:
        print(f"Unexpected error [{type(exc).__name__}]: {str(exc)[:300]}")
        return None


def show_response(title, response, max_chars=600):
    """응답 텍스트를 너무 길지 않게 출력한다."""
    if response is None:
        print("응답이 없습니다.")
        return

    text = response.text or ""
    print(f"[{title}]")
    print(text[:max_chars])


def show_usage(response):
    """응답 객체에서 토큰 사용량과 종료 이유를 꺼내 출력한다."""
    if response is None:
        return

    usage = response.usage_metadata
    candidate = response.candidates[0] if response.candidates else None

    print("\n[토큰 사용량]")
    print(f"입력 토큰: {usage.prompt_token_count}")
    print(f"출력 토큰: {usage.candidates_token_count}")
    print(f"총 토큰:   {usage.total_token_count}")

    if candidate:
        print(f"중단 이유: {candidate.finish_reason}")

## 2. 첫 번째 Gemini API 호출

`client.models.generate_content()`는 가장 기본적인 텍스트 생성 메서드입니다.

- `model`: 사용할 모델 이름
- `contents`: 모델에게 보낼 질문 또는 지시문
- `config`: 출력 길이, 창의성 같은 옵션

In [11]:
response = call_gemini(
    "첫 번째 Gemini API 호출",
    lambda: client.models.generate_content(
        model=MODEL,
        contents="안녕하세요! 한 문장으로 자기소개를 해주세요.",
        config=types.GenerateContentConfig(max_output_tokens=120),
    ),
)

show_response("Gemini 응답", response)


[API 요청 1] 첫 번째 Gemini API 호출
--------------------------------------------------
APIError [503 UNAVAILABLE]: This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.
응답이 없습니다.


## 3. 응답 구조 분석

Gemini 응답에서 가장 자주 보는 값은 `response.text`입니다.

하지만 서비스 개발에서는 토큰 사용량과 종료 이유도 중요합니다.

- `usage_metadata.prompt_token_count`: 입력 토큰 수
- `usage_metadata.candidates_token_count`: 출력 토큰 수
- `usage_metadata.total_token_count`: 총 토큰 수
- `candidates[0].finish_reason`: 응답이 끝난 이유

In [12]:
response = call_gemini(
    "응답 구조 분석",
    lambda: client.models.generate_content(
        model=MODEL,
        contents="Python의 장점 3가지를 짧은 글머리표로 나열해주세요.",
        config=types.GenerateContentConfig(max_output_tokens=180),
    ),
)

show_response("응답 텍스트", response)
show_usage(response)


[API 요청 2] 응답 구조 분석
--------------------------------------------------
[응답 텍스트]
Python의 주요 장점 3가지는 다음과 같습니다.

* **높은 가독성과 간결함:** 문법이 영어와 유사하여 배우기 쉽고, 코드의 길이가 짧아 유지보수가 용이합니다.
* **방대한 라이브러리와 생태계:** 데이터 분석, 인공지능, 웹 개발 등 다양한 분야에서 활용 가능한 풍부한 라이브러리를 제공합니다.
* **높은 생산성:** 인터프리터 언어로서 코드 작성 즉시 실행 결과 확인이 가능해 개발 속도가 매우 빠릅니다.

[토큰 사용량]
입력 토큰: 20
출력 토큰: 120
총 토큰:   140
중단 이유: FinishReason.STOP


## 4. 시스템 지시문

`system_instruction`은 모델에게 역할과 답변 방식을 미리 알려주는 설정입니다.

아래 셀에서 `role`과 `question`을 바꿔 보면서 답변 톤이 어떻게 달라지는지 확인합니다.

In [13]:
role = "친절한 한국어 선생님"
question = "영어 문장 'The quick brown fox'를 초보자에게 설명해주세요."

response = call_gemini(
    "시스템 지시문 실습",
    lambda: client.models.generate_content(
        model=MODEL,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=f"당신은 {role}입니다. 항상 그 역할에 맞게 답변하세요.",
            max_output_tokens=220,
        ),
    ),
)

print(f"역할: {role}")
print(f"질문: {question}")
show_response("답변", response)


[API 요청 3] 시스템 지시문 실습
--------------------------------------------------
역할: 친절한 한국어 선생님
질문: 영어 문장 'The quick brown fox'를 초보자에게 설명해주세요.
[답변]
안녕하세요! 여러분의 한국어 선생님입니다. 영어 공부를 시작하셨군요! 아주 반갑습니다.

'The quick brown fox'는 영어권에서 아주 유명한 문장이에요. 단순히 뜻을 떠나서 아주 재미있는 이유가 있거든요. 선생님이 차근차근 설명해 줄게요.

### 1. 단어별 뜻 풀이
먼저 각 단어의 뜻을 알아볼까요?

*   **The**: (특정한 대상을 가리킬 때 쓰는) 그
*   **quick**: 빠른
*   **brown**: 갈색의
*   **fox**: 여우

다 합치면 **"그 빠른 갈색 여우"**라는 뜻이 됩니다. 아주 쉽죠?

### 2. 왜 이 문장이 유명할까요?
이 문장은 단순히 여우를 설명하려고 만든 게 아니에요. 뒤에 이어지는 문장까지 함께 보면 그 이유를 알 수 있어요.

> **"The quick brown fox jumps over the lazy dog."**
> (그


### 역할 바꿔보기

수업 중 시간이 있으면 아래 예시 중 하나를 골라 위 셀의 `role`, `question`에 넣어 실행해 봅니다.

In [14]:
role_examples = [
    ("친절한 한국어 선생님", "영어 문장 'The quick brown fox'를 설명해주세요."),
    ("엄격한 코드 리뷰어", "아래 코드를 검토해주세요:\ndef add(a, b):\n  return a+b"),
    ("5살 아이에게 설명하는 과학자", "블랙홀이 뭔가요?"),
]

for index, (example_role, example_question) in enumerate(role_examples, start=1):
    print(f"{index}. {example_role} -> {example_question[:40]}")

1. 친절한 한국어 선생님 -> 영어 문장 'The quick brown fox'를 설명해주세요.
2. 엄격한 코드 리뷰어 -> 아래 코드를 검토해주세요:
def add(a, b):
  return a
3. 5살 아이에게 설명하는 과학자 -> 블랙홀이 뭔가요?


## 5. 토큰 사전 카운팅

토큰은 AI 모델이 텍스트를 나누어 읽는 단위입니다.

긴 입력을 보내기 전에 `count_tokens()`로 대략적인 입력 크기를 확인할 수 있습니다.

In [15]:
texts = [
    "Hello",
    "안녕하세요",
    "FastAPI는 Python 기반의 고성능 웹 프레임워크로, 자동 문서화와 타입 안전성을 제공합니다.",
    "Explain the difference between synchronous and asynchronous programming in Python with examples.",
]

for text in texts:
    token_response = call_gemini(
        f"토큰 계산: {text[:20]}",
        lambda text=text: client.models.count_tokens(
            model=MODEL,
            contents=text,
        ),
    )

    if token_response:
        print(f"{token_response.total_tokens:4d} tokens | {text[:70]}")


[API 요청 4] 토큰 계산: Hello
--------------------------------------------------
   2 tokens | Hello

[API 요청 5] 토큰 계산: 안녕하세요
--------------------------------------------------
   2 tokens | 안녕하세요

[API 요청 6] 토큰 계산: FastAPI는 Python 기반의 
--------------------------------------------------
  27 tokens | FastAPI는 Python 기반의 고성능 웹 프레임워크로, 자동 문서화와 타입 안전성을 제공합니다.

[API 요청 7] 토큰 계산: Explain the differen
--------------------------------------------------
  14 tokens | Explain the difference between synchronous and asynchronous programmin


## 6. 에러 처리

API를 쓰는 코드는 실패 상황도 함께 다뤄야 합니다.

아래 셀은 일부러 잘못된 모델명을 사용해서 `APIError`가 어떻게 출력되는지 확인합니다.

In [16]:
bad_response = call_gemini(
    "잘못된 모델명으로 에러 확인",
    lambda: client.models.generate_content(
        model="gemini-nonexistent-model",
        contents="안녕",
    ),
)

print("bad_response =", bad_response)


[API 요청 8] 잘못된 모델명으로 에러 확인
--------------------------------------------------
APIError [404 NOT_FOUND]: models/gemini-nonexistent-model is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
bad_response = None


### 자주 만나는 오류

| 코드 | 의미 | 확인할 것 |
|------|------|-----------|
| 400 | 요청 형식 오류 | 모델명, 파라미터, 입력 형식 |
| 403 | 인증 또는 권한 문제 | API 키, 프로젝트 권한 |
| 429 | 요청 한도 초과 | 잠시 대기, 요청 수 줄이기 |
| 500 | 서버 내부 오류 | 잠시 후 재시도 |

## 7. 사용 가능한 모델 목록 조회

현재 계정에서 사용할 수 있는 모델을 확인합니다.

모델 목록은 시간이 지나면서 바뀔 수 있으므로, 수업 자료의 표보다 실제 API 결과를 우선 확인합니다.

In [17]:
models_response = call_gemini(
    "모델 목록 조회",
    lambda: list(client.models.list()),
)

if models_response:
    gemini_models = [m for m in models_response if "gemini" in m.name.lower()]

    for model in gemini_models[:10]:
        name = model.name.split("/")[-1] if "/" in model.name else model.name
        print(name)

    if len(gemini_models) > 10:
        print(f"... 외 {len(gemini_models) - 10}개")


[API 요청 9] 모델 목록 조회
--------------------------------------------------
gemini-2.5-flash
gemini-2.5-pro
gemini-2.5-flash-preview-tts
gemini-2.5-pro-preview-tts
gemini-flash-latest
gemini-flash-lite-latest
gemini-pro-latest
gemini-2.5-flash-lite
gemini-2.5-flash-image
gemini-3-flash-preview
... 외 35개
